# Gradient descent variants (batch, mini-batch, SGD)

Batch, mini-batch, and SGD trade gradient accuracy against update frequency.

Gradient descent variants turn empirical risk into repeated parameter updates. Batch uses the full sample, mini-batch trades exactness for throughput, and SGD makes the noisiest but most frequent updates. Save a copy to Drive to edit.

In [ ]:

import math
import random
import numpy as np
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_diabetes
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn.datasets import make_moons
from sklearn.datasets import make_regression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

np.random.seed(7)
random.seed(7)


def clf_ladder():
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def reg_ladder():
    rungs = []

    x1 = np.array([[0.0], [1.0], [2.0], [3.0], [4.0], [5.0]])
    y1 = np.array([1.0, 2.9, 5.2, 7.1, 8.9, 11.2])
    rungs.append(("D1 hand 3-point line plus checks", x1, y1))

    x2, y2 = make_regression(n_samples=220, n_features=3, n_informative=3, noise=8.0, random_state=2)
    rungs.append(("D2 clean make_regression", x2, y2))

    rng = np.random.default_rng(3)
    x3, y3 = make_regression(n_samples=260, n_features=5, n_informative=4, noise=18.0, random_state=3)
    outliers = rng.choice(np.arange(y3.size), size=18, replace=False)
    y3[outliers] = y3[outliers] + rng.normal(0.0, 240.0, size=outliers.size)
    rungs.append(("D3 noisy/outlier regression", x3, y3))

    diabetes = load_diabetes()
    rungs.append(("D4 Diabetes (real, 10-D)", diabetes.data, diabetes.target))

    poly = PolynomialFeatures(degree=2, include_bias=False)
    x5 = poly.fit_transform(diabetes.data)
    y5 = diabetes.target.copy()
    rungs.append(("D5 Diabetes expanded interactions (real, 65-D)", x5, y5))

    return rungs


def reg_mse(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    mse = mean_squared_error(y_te, preds)
    r2 = r2_score(y_te, preds)
    return float(mse), float(r2)


def lesson_score(losses, cost, alternative):
    raw = sum(losses) / len(losses)
    score = raw + cost
    gap = alternative - score
    relative_gap = gap / alternative
    stabilized = 0.80 * score
    return raw, score, gap, relative_gap, stabilized


def print_ladder_preview(rungs):
    for name, X, y in rungs:
        unique = np.unique(y)
        label = "classes=" + str(unique.size) if unique.size <= 20 else "target_range=" + str((float(np.min(y)), float(np.max(y))))
        print(f"{name:42s} X={X.shape} {label}")
        print("sample X", np.round(X[:3], 3))
        print("sample y", np.round(y[:6], 3))


def plot_regression_results(rungs, fitted, mses):
    fig, axes = plt.subplots(2, 5, figsize=(18, 6))
    for idx, (name, X, y) in enumerate(rungs):
        ax = axes[0, idx]
        x_axis = np.arange(y.size)
        order = np.argsort(X[:, 0])
        preds = fitted[idx]
        ax.scatter(x_axis[:80], y[:80], s=12, alpha=0.65, label="actual")
        ax.scatter(x_axis[:80], preds[:80], s=12, alpha=0.65, label="fit")
        ax.set_title(name.split(" (")[0], fontsize=9)
        ax.tick_params(labelsize=7)
        if idx == 0:
            ax.legend(fontsize=7)
    axes[1, 0].plot(np.arange(1, 6), mses, marker="o")
    axes[1, 0].set_xticks(np.arange(1, 6))
    axes[1, 0].set_xlabel("rung")
    axes[1, 0].set_ylabel("held-out MSE")
    axes[1, 0].set_title("MSE vs complexity")
    for ax in axes[1, 1:]:
        ax.axis("off")
    fig.tight_layout()
    plt.show()


def plot_classification_results(rungs, build_and_predict, accs, title):
    fig, axes = plt.subplots(2, 5, figsize=(18, 6))
    for idx, (name, X, y) in enumerate(rungs):
        ax = axes[0, idx]
        scaler = StandardScaler()
        x_scaled = scaler.fit_transform(X)
        x_vis = x_scaled[:, :2]
        x_tr, x_te, y_tr, y_te = train_test_split(x_vis, y, test_size=0.4, random_state=0, stratify=y)
        preds = build_and_predict(x_tr, y_tr, x_te)
        ax.scatter(x_te[:, 0], x_te[:, 1], c=preds, cmap="tab10", s=15, alpha=0.75)
        ax.set_title(name.split(" (")[0], fontsize=9)
        ax.tick_params(labelsize=7)
    axes[1, 0].plot(np.arange(1, 6), accs, marker="o")
    axes[1, 0].set_xticks(np.arange(1, 6))
    axes[1, 0].set_ylim(0.0, 1.05)
    axes[1, 0].set_xlabel("rung")
    axes[1, 0].set_ylabel("held-out accuracy")
    axes[1, 0].set_title(title)
    for ax in axes[1, 1:]:
        ax.axis("off")
    fig.tight_layout()
    plt.show()


## The concept, built once on D1

The lesson formula is

$$w_{t+1}=w_t-\eta\frac1{|B_t|}\sum_{i\in B_t}\nabla_w\ell_i(w_t)$$

We first reproduce the lesson arithmetic exactly: losses 0.191, 0.083, and 0.420 give $R_S=0.694/3=0.231$. Adding cost 0.060 gives score 0.291; the alternative 0.343 leaves gap 0.052. The stabilized score is $0.80\times 0.291=0.233$.

In [ ]:
losses = np.array([0.191, 0.083, 0.42])
cost = 0.060
alternative = 0.343
raw_unrounded, _, _, _, _ = lesson_score(losses, cost, alternative)
raw = 0.231
score = raw + cost
gap = alternative - score
relative_gap = gap / alternative
stabilized = 0.80 * score

assert round(float(losses.sum()), 3) == 0.694
assert round(raw_unrounded, 3) == 0.231
assert round(raw, 3) == 0.231
assert round(score, 3) == 0.291
assert round(gap, 3) == 0.052
assert round(relative_gap, 3) == 0.152

print("raw risk", round(raw, 3))
print("score", round(score, 3))
print("gap", round(gap, 3))
print("relative gap", round(relative_gap, 3))
print("stabilized", round(stabilized, 3))

### Build a reusable optimizer

The same linear model can be trained with full-batch, mini-batch, or one-example SGD updates. The code below implements the real gradient of squared error plus ridge shrinkage; only the batch schedule changes.

In [ ]:
def gradient_descent_variants_batch_mini_batch_method(X, y, variant="mini", lr=0.05, epochs=400, batch_size=32, ridge=0.001):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    design = np.c_[np.ones(X.shape[0]), X]
    weights = np.zeros(design.shape[1])
    rng = np.random.default_rng(11)
    losses = []

    for epoch in range(epochs):
        if variant == "batch":
            batches = [np.arange(design.shape[0])]
        elif variant == "sgd":
            order = rng.permutation(design.shape[0])
            batches = [np.array([idx]) for idx in order]
        else:
            order = rng.permutation(design.shape[0])
            batches = [order[start:start + batch_size] for start in range(0, design.shape[0], batch_size)]

        for batch in batches:
            xb = design[batch]
            yb = y[batch]
            error = xb @ weights - yb
            grad = xb.T @ error / batch.size
            grad[1:] = grad[1:] + ridge * weights[1:]
            weights = weights - lr * grad

        full_error = design @ weights - y
        losses.append(float(np.mean(full_error ** 2)))

    def predict(X_new):
        X_new = np.asarray(X_new, dtype=float)
        return np.c_[np.ones(X_new.shape[0]), X_new] @ weights

    return weights, np.array(losses), predict

X_demo = np.array([[0.0], [1.0], [2.0]])
y_demo = np.array([1.0, 3.0, 5.0])
weights, trace, predict_demo = gradient_descent_variants_batch_mini_batch_method(X_demo, y_demo, variant="batch", lr=0.08, epochs=1200, ridge=0.0)
assert trace[-1] < 0.001
print("learned weights", np.round(weights, 3))
print("D1 predictions", np.round(predict_demo(X_demo), 3))

## The dataset ladder

In [ ]:
regression_rungs = reg_ladder()
print_ladder_preview(regression_rungs)

## Run the same method across D1-D5

In [ ]:
variant_metrics = []
fitted_values = []

for name, X, y in regression_rungs:
    def build_and_predict(x_tr, y_tr, x_te):
        weights, trace, predict = gradient_descent_variants_batch_mini_batch_method(x_tr, y_tr, variant="mini", lr=0.03, epochs=650, batch_size=32, ridge=0.01)
        return predict(x_te)

    mse, r2 = reg_mse(build_and_predict, X, y)
    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(X)
    weights, trace, predict = gradient_descent_variants_batch_mini_batch_method(x_scaled, y, variant="mini", lr=0.03, epochs=650, batch_size=32, ridge=0.01)
    preds_all = predict(x_scaled)
    fitted_values.append(preds_all)
    variant_metrics.append((name, mse, r2))

print("rung | MSE | R2")
for idx, (name, mse, r2) in enumerate(variant_metrics, start=1):
    print(f"D{idx} | {mse:.3f} | {r2:.3f} | {name}")

mses = [row[1] for row in variant_metrics]

## Results visualization

In [ ]:
plot_regression_results(regression_rungs, fitted_values, mses)

## Pitfall on D5: optimizing raw MSE while forgetting the lesson cost

In [ ]:
name, X, y = regression_rungs[-1]
raw_scores = {}
with_cost_scores = {}

for variant, cost_add in [("batch", 0.060), ("mini", 0.030), ("sgd", 0.000)]:
    def build_and_predict(x_tr, y_tr, x_te, variant=variant):
        weights, trace, predict = gradient_descent_variants_batch_mini_batch_method(x_tr, y_tr, variant=variant, lr=0.01, epochs=300, batch_size=24, ridge=0.05)
        return predict(x_te)

    mse, r2 = reg_mse(build_and_predict, X, y)
    raw_scores[variant] = mse
    with_cost_scores[variant] = mse * (1.0 + cost_add)

raw_winner = min(raw_scores, key=raw_scores.get)
cost_winner = min(with_cost_scores, key=with_cost_scores.get)
print("raw MSE scores", {key: round(value, 2) for key, value in raw_scores.items()})
print("cost-aware scores", {key: round(value, 2) for key, value in with_cost_scores.items()})
print("raw winner", raw_winner)
print("cost-aware winner", cost_winner)
print("lesson gap check", round(0.343 - 0.291, 3))
assert round(0.343 - 0.291, 3) == 0.052

## Evaluate it + practice

- Report the held-out MSE beside a no-skill baseline such as majority-class accuracy or mean-target MSE.
- Sanity check that shuffling labels or targets destroys the useful signal.
- Ablation: turn mini-batching into single full-batch updates or one-example SGD and compare convergence plus D5 MSE.
- Watch failure signals: validation instability, scale leakage, and a score that improves only when the lesson cost is ignored.
- Recompute the lesson raw risk, cost, gap, relative gap, and stabilized score whenever you compare settings.

Practice 1: change one hyperparameter and rerun the D1-D5 table.

Practice 2: add a label/target shuffle baseline and explain the drop.

Practice 3: repeat the pitfall cell with a different cost value.